In [ ]:
PROJECT_RANDOM_STATE = 2627
print("PROJECT_RANDOM_STATE =", PROJECT_RANDOM_STATE)

# Practice 4 — Challenge B (storage): regression tree

**Contents:** P4.3, the two rules of P4.4 (from the Challenge B tree) and the Challenge B part of P4.5.

## 1. Setup

In [ ]:
import json
import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn import metrics

# Paths are relative to the notebooks/ folder, where this file is run.
CYBER_DIR = "../datos/cyber"
STORAGE_DIR = "../datos/storage"
FOLDS_DIR = "../folds"

## 2. Data

In [ ]:
# Same names as in cyber, so that R5 can join the two blocks.
dev = pd.read_csv(STORAGE_DIR + "/storage_development.csv")
storage_groups = pd.read_csv(STORAGE_DIR + "/storage_groups.csv")
storage_folds = pd.read_csv(FOLDS_DIR + "/storage_folds.csv")

# raid is a nominal label, even though its levels contain numbers.
NOMINAL = ["load_type", "io_type", "raid", "device_type"]
NUMERIC = ["block_size", "n_jobs", "iodepth", "read_fraction", "n_disks"]
FEATURES = NOMINAL + NUMERIC

# The CSV has a different column order: compare the sorted names.
expected_columns = ["row_id"] + FEATURES + ["latency_ns"]
assert sorted(dev.columns) == sorted(expected_columns), "Unexpected storage schema"
assert "iops" not in dev.columns, "Excluded column iops must not be present"
X = dev[FEATURES].copy()
latency_ns = dev["latency_ns"].copy()

assert not dev.isna().any().any(), "Missing values in development data"
assert np.isfinite(X[NUMERIC].to_numpy()).all(), "Non-finite numeric predictors"
assert np.isfinite(latency_ns).all(), "Non-finite target values"
assert (latency_ns > 0).all(), "Latency must be strictly positive"
z = np.log10(latency_ns)

for column in ["row_id", "config_group", "feature_group", "fold_id", "latency_ns", "iops"]:
    assert column not in X.columns, "Forbidden predictor: " + column
assert X.shape[1] == 9, "Expected nine source predictors"

print("Development table:", dev.shape)
print("Predictors:", X.shape, "(", len(NOMINAL), "nominal and", len(NUMERIC), "numeric )")
print("Target: z = log10(latency_ns)")

In [ ]:
# Check that the three tables are aligned before using positions.
assert list(storage_groups.columns) == ["row_id", "config_group"], "Unexpected group schema"
assert list(storage_folds.columns) == ["row_id", "fold_id"], "Unexpected fold schema"
assert dev["row_id"].is_unique, "Duplicate development row_id"
assert dev["row_id"].equals(storage_groups["row_id"]), "Group rows are not aligned"
assert dev["row_id"].equals(storage_folds["row_id"]), "Fold rows are not aligned"
assert sorted(storage_folds["fold_id"].unique()) == [0, 1, 2, 3, 4], "Expected five folds"

# Read the official assignment: the folds are never rebuilt.
folds = []
fold_rows = []
for fold_id in range(5):
    train_idx = np.flatnonzero(storage_folds["fold_id"].to_numpy() != fold_id)
    val_idx = np.flatnonzero(storage_folds["fold_id"].to_numpy() == fold_id)

    # The replicates of a configuration always stay together.
    train_groups = set(storage_groups.iloc[train_idx]["config_group"])
    val_groups = set(storage_groups.iloc[val_idx]["config_group"])
    assert train_groups.isdisjoint(val_groups), "Group leakage in fold " + str(fold_id)
    assert len(train_idx) + len(val_idx) == len(dev), "Incomplete partition"

    folds.append((train_idx, val_idx))
    fold_rows.append({"fold_id": fold_id, "training_rows": len(train_idx),
                      "validation_rows": len(val_idx)})

print("Configuration groups:", storage_groups["config_group"].nunique())
print(pd.DataFrame(fold_rows).to_string(index=False))
print("All five canonical folds passed the group separation checks.")

The storage development table has 12,190 rows and 9 source predictors (4 nominal and 5 numeric), grouped in 2,438 configurations. The five canonical folds are read from `../folds/storage_folds.csv`, and the checks above confirm that the replicates of a configuration never fall on both sides of a split. Identifiers, grouping metadata, fold assignments and the target are kept outside `X`; `iops` is absent.

The target is `z = log10(latency_ns)`, as fixed in P2.3. The original `latency_ns` is kept for physical-scale evaluation. R3 must invert its predictions with `10 ** z_hat` before computing any error in nanoseconds.

## 3. Representation for the tree (statement, Section 2)

**Scaling.** No scaler. The reasoning is the same as in cyber: a tree tests `x <= t` on one column, and an increasing transformation moves the threshold without changing which rows go to each side. The five numeric predictors (`block_size`, `n_jobs`, `iodepth`, `read_fraction`, `n_disks`) pass through in their original units, so a threshold on `n_disks` is read directly as a number of disks. P3 standardised them; P4 does not. The target transformation `z = log10(latency_ns)` is separate from this preprocessor and does not change.

**Categorical encoding.** `OneHotEncoder(handle_unknown="ignore", sparse_output=False)` with every level kept. P3 removed `random`, `write`, `4+1` and `nvme`; a tree needs no reference level, and keeping them lets it ask about any single level, such as `device_type_hdd`. `raid` remains a nominal label: its levels are schemes, not ordered numbers.

**Unseen levels.** With `handle_unknown="ignore"`, an unseen level would be encoded as zeros for its predictor, meaning "none of the known levels". The checks below show that no validation row has an unseen level in any fold.

### Correspondence with P3

In [ ]:
# Levels of each nominal predictor in the development table, and the P3 mapping.
P3_DROPPED_LEVELS = ["random", "write", "4+1", "nvme"]

mapping_rows = []
for column, dropped in zip(NOMINAL, P3_DROPPED_LEVELS):
    levels = sorted(X[column].unique())
    assert dropped in levels, "P3 reference level not found: " + dropped
    mapping_rows.append({"predictor": column, "levels": len(levels),
                         "P3_dropped_level": dropped,
                         "P3_indicator_columns": len(levels) - 1,
                         "P4_indicator_columns": len(levels)})

mapping = pd.DataFrame(mapping_rows)
print(mapping.to_string(index=False))
print("Total indicator columns, P3:", mapping["P3_indicator_columns"].sum())
print("Total indicator columns, P4:", mapping["P4_indicator_columns"].sum())

P3 dropped `load_type=random`, `io_type=write`, `raid=4+1` and `device_type=nvme`. P4 keeps them as columns, and it does not scale the numeric predictors. The four predictors have 2, 2, 6 and 2 levels, so the indicator columns go from 8 in P3 to 12 in P4. The check below confirms, on every fold, that each P3 indicator column has the same values as the P4 column of the same level.

In [ ]:
def make_preprocessor():
    # One-hot with every level kept; unseen levels become all zeros.
    nominal_steps = OneHotEncoder(handle_unknown="ignore", sparse_output=False)

    # Numeric predictors pass through unchanged: no scaler.
    preprocessor = ColumnTransformer([
        ("cat", nominal_steps, NOMINAL),
        ("num", "passthrough", NUMERIC),
    ])
    # Returned unfitted so that R3 puts it inside its Pipeline.
    return preprocessor

In [ ]:
# Check the block inside each canonical fold: fitted on training rows only.
check_rows = []
for fold_id, (train_idx, val_idx) in enumerate(folds):
    preprocessor = make_preprocessor()
    assert not hasattr(preprocessor, "transformers_"), "Expected an unfitted transformer"
    X_train = X.iloc[train_idx]
    X_valid = X.iloc[val_idx]
    X_train_t = preprocessor.fit_transform(X_train)
    X_valid_t = preprocessor.transform(X_valid)

    encoder = preprocessor.named_transformers_["cat"]
    assert encoder.drop_idx_ is None, "No level may be dropped"

    # Count the columns and the validation levels that training never saw.
    n_cat = 0
    unseen_per_row = np.zeros(len(X_valid), dtype=int)
    for column, categories in zip(NOMINAL, encoder.categories_):
        assert list(categories) == sorted(X_train[column].unique()), "Wrong vocabulary"
        n_cat = n_cat + len(categories)
        unseen_per_row = unseen_per_row + (~X_valid[column].isin(categories)).to_numpy()
    n_num = len(NUMERIC)

    feature_names = preprocessor.get_feature_names_out()
    assert X_train_t.shape[1] == len(feature_names), "Missing feature names"
    assert X_valid_t.shape[1] == len(feature_names), "Inconsistent columns"
    assert len(feature_names) == n_cat + n_num, "Unexpected number of columns"
    assert np.isfinite(X_train_t).all(), "Non-finite training values"
    assert np.isfinite(X_valid_t).all(), "Non-finite validation values"

    # Numeric columns keep their original values: nothing was scaled.
    assert np.array_equal(X_train_t[:, n_cat:], X_train[NUMERIC].to_numpy()), "Train values changed"
    assert np.array_equal(X_valid_t[:, n_cat:], X_valid[NUMERIC].to_numpy()), "Valid values changed"

    # One indicator per predictor is 1 in training; an unseen level gives 0.
    train_ones = X_train_t[:, :n_cat].sum(axis=1)
    valid_ones = X_valid_t[:, :n_cat].sum(axis=1)
    assert (train_ones == len(NOMINAL)).all(), "Each training row needs one level per predictor"
    assert (valid_ones == len(NOMINAL) - unseen_per_row).all(), "Unseen levels must give zeros"

    if fold_id == 0:
        names_fold_0 = list(feature_names)

    check_rows.append({"fold_id": fold_id, "output_features": len(feature_names),
                       "indicator_columns": n_cat, "numeric_columns": n_num,
                       "unseen_validation_rows": int((unseen_per_row > 0).sum())})

print(pd.DataFrame(check_rows).to_string(index=False))
print("Fold 0 output names:", names_fold_0)
print("Preprocessing checks passed on all five folds.")

In [ ]:
# Every P3 indicator column must equal the P4 column of the same level.
for fold_id, (train_idx, val_idx) in enumerate(folds):
    X_train = X.iloc[train_idx]
    X_valid = X.iloc[val_idx]

    p4 = make_preprocessor()
    p4.fit(X_train)
    p4_names = list(p4.named_transformers_["cat"].get_feature_names_out(NOMINAL))
    p4_valid = p4.named_transformers_["cat"].transform(X_valid[NOMINAL])

    p3 = OneHotEncoder(drop=P3_DROPPED_LEVELS, handle_unknown="ignore", sparse_output=False)
    p3.fit(X_train[NOMINAL])
    p3_names = list(p3.get_feature_names_out(NOMINAL))
    p3_valid = p3.transform(X_valid[NOMINAL])

    for position, name in enumerate(p3_names):
        assert name in p4_names, "P3 column missing in P4: " + name
        same_column = p4_valid[:, p4_names.index(name)]
        assert np.array_equal(p3_valid[:, position], same_column), "Different values: " + name
    print("Fold", fold_id, ": P3 columns", len(p3_names), "are all in P4 columns", len(p4_names))

In [ ]:
# The target representation is unchanged from P2.3 and P3.
assert np.isfinite(z).all(), "Non-finite log target"
assert np.allclose(10 ** z, latency_ns), "Target inversion does not recover latency"
print("Target check passed: 10 ** z recovers latency_ns.")

**Checks.** All checks pass on the five canonical folds. The transformed matrix has 17 columns in every fold (12 indicators and 5 numeric), training and validation have the same columns, the numeric columns keep their original values, and no validation row has a level that training never saw. The target check confirms that `10 ** z` recovers `latency_ns`. These are preprocessing checks, not model results.

`make_preprocessor()` returns a new, unfitted transformer. R3 must place it inside the model `Pipeline`, fitted only on each fold's training rows. The output names (`cat__device_type_hdd`, `num__block_size`) can be used to read rules: a split on `cat__device_type_hdd` separates the hdd devices from the others.

## 4. Canonical folds and metrics

In [ ]:
fold_table = pd.read_csv("../folds/storage_folds.csv")
#the positions only make sense if the fold file lists the rows in the same order as dev
assert np.array_equal(fold_table["row_id"].to_numpy(), dev["row_id"].to_numpy())
fold_id = fold_table["fold_id"].to_numpy()

#lets build the folds (this list is what R3 passes as cv=folds)
folds = []
for f in range(5):
    train_idx = np.where(fold_id != f)[0]
    val_idx = np.where(fold_id == f)[0]
    folds.append((train_idx, val_idx))

print(len(folds))

In [ ]:
grps = storage_groups["config_group"].to_numpy()
#the same for the groups file
assert np.array_equal(storage_groups["row_id"].to_numpy(), dev["row_id"].to_numpy())
#we check that the 5 replicates of a configuration never end up split between train and validation
for train_idx, val_idx in folds:
    grps_train = set(grps[train_idx])
    grps_val = set(grps[val_idx])
    assert len(grps_train & grps_val) == 0 #the two sets of groups must not share any group
    print(len(train_idx), len(val_idx))

print("no config group is in both training and validation")

In [ ]:
def spread(values):
    res = np.max(values) - np.min(values)
    
    return res

#storage metrics on the validation rows of one fold
#the first three are computed on z; MAE_ns only after turning z_pred back into nanoseconds
def storage_metrics(z_true, z_pred, latency_true):
    mae = metrics.mean_absolute_error(z_true, z_pred)
    rmse = np.sqrt(metrics.mean_squared_error(z_true, z_pred))
    r2 = metrics.r2_score(z_true, z_pred)
    mae_ns = metrics.mean_absolute_error(latency_true, 10 ** z_pred)

    return [mae, rmse, r2, mae_ns]

# P4.3 — Tree regression

In [ ]:
# Imports for this part
import time     # to measure how long the search and each fit take
from sklearn.pipeline import Pipeline    # joins preprocessing and tree in one object that is fitted as a whole 
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score   # MAE, MSE (its root is the RMSE) and R2
from sklearn.model_selection import GridSearchCV    # tries every combination of the grid on the folds
from sklearn.tree import DecisionTreeRegressor    # the regression tree of this practice

In [ ]:
# Declare the grid values for this task
def make_tree():
    # A new, unfitted pipeline every time: each fold gets a fresh copy,
    # so nothing learned in one fold leaks into another
    return Pipeline([
        ("preprocessing", make_preprocessor()),  # one-hot nominal columns, pass numeric ones unchanged
        ("tree", DecisionTreeRegressor(random_state=PROJECT_RANDOM_STATE)),  # learns to predict z; seed fixed for ties
    ])

# Check 1: how many rows each configuration has
rows_per_config = dev.groupby(FEATURES).size()
print("Number of configurations:", len(rows_per_config))    # one entry per different configuration
print("Configurations by number of rows:")
print(rows_per_config.value_counts())          # how many configurations have 1 row, 2 rows...

# Check 2: in how many different folds each configuration appears (fold_id is only used for this check, never as a predictor)
check = dev[FEATURES].copy()        # a copy, so that dev itself is not changed
check["fold_id"] = storage_folds["fold_id"]                        # same row order as dev 
folds_per_config = check.groupby(FEATURES)["fold_id"].nunique()     # number of different folds per configuration
print("Configurations by number of different folds:")
print(folds_per_config.value_counts())      # if every configuration has 1, none is split between folds

# Declared grid: names start with "tree__" because the tree is the "tree" step of the pipeline
param_grid = {
    "tree__max_depth": [4, 6, 8, 10, 12, None],      # None = no depth limit
    "tree__min_samples_leaf": [5, 10, 20, 40],       # multiples of 5 replicates
    "tree__ccp_alpha": [0.0, 0.00001, 0.0001, 0.001],# pruning strength, in squared units of z
}

# Numbers used in the Markdown cells
# Leaf size in rows -> leaf size in configurations
for leaf in param_grid["tree__min_samples_leaf"]:
    configs = leaf // 5          # each configuration has 5 rows; // is integer division
    print("min_samples_leaf =", leaf, "-> at least", configs, "configurations per leaf")

# scale of z, to judge whether the ccp_alpha values are large or small
# ddof=0 divides by the number of rows, like the squared error that the tree minimises
print("Variance of z:", round(z.var(ddof=0), 4))     

# Size of the search
n_depth = len(param_grid["tree__max_depth"])     # number of max_depth values
n_leaf = len(param_grid["tree__min_samples_leaf"])       # number of min_samples_leaf values
n_alpha = len(param_grid["tree__ccp_alpha"])     # number of ccp_alpha values
n_combinations = n_depth * n_leaf * n_alpha      # every combination is tried
n_fits = n_combinations * len(folds)        # each combination is fitted once per fold
print("Combinations:", n_combinations, "| fits:", n_fits)

**Replicates** 

The 12190 rows are 2438 configurations of the 9 predictors, each with exactly 5 rows, and every configuration appears in a single fold (both checks above). This has two consequences:
- A configuration is never in training and validation at the same time, so validation measures the error on configurations the tree has not seen.
- The 5 replicates of a configuration have identical predictors, so no split can separate them. Every leaf holds whole configurations, so at least 5 rows, and any `min_samples_leaf` from 1 to 5 gives the same tree. The fold 0 check below confirms it.

**Declared grid and why.**

- `max_depth` in {4, 6, 8, 10, 12, None}: from shallow trees that are easy to read to a tree with no depth limit (`None`), so the search shows where more depth stops helping on validation.
- `min_samples_leaf` in {5, 10, 20, 40}: at least 1, 2, 4 and 8 configurations per leaf. The value 5 stands for every value from 1 to 5, which all give the same tree; 40 forces leaves of at least 8 configurations.
- `ccp_alpha` in {0.0, 1e-05, 0.0001, 0.001}: in regression it is measured in squared units of z, whose variance is 0.7056. 0.0 means no pruning, and each non-zero value is ten times the previous one.
- 96 combinations, 480 fits on the shared canonical folds, scored with MAE(z), the primary metric of Challenge B.
- `random_state` is fixed because ties between equally good splits are broken with the estimator's random number generator, so the seed can change the fitted tree.
- Numeric columns are not scaled: a tree splits on one column at a time, and a positive rescaling moves the threshold without changing the partition it induces.

In [ ]:
# Fold 0 only, to read the data before the search (nothing here chooses the hyperparameters)
train_idx = folds[0][0]   # positions of the training rows of fold 0
val_idx = folds[0][1]     # positions of the validation rows of fold 0

# 1) Unlimited tree: the defaults max_depth=None, min_samples_leaf=1, ccp_alpha=0: the tree grows as much as it can
full_tree = make_tree()
full_tree.fit(X.iloc[train_idx], z.iloc[train_idx])  # learn only from the training rows
# Error on the validation configurations, which the tree has never seen
full_mae = mean_absolute_error(z.iloc[val_idx], full_tree.predict(X.iloc[val_idx]))
# named_steps["tree"] is the tree inside the pipeline; get_n_leaves() counts its leaves
print("Unlimited tree, fold 0 | leaves:", full_tree.named_steps["tree"].get_n_leaves(),
     "| configurations in training:", len(train_idx) // 5,  # 5 rows per configuration
     "| validation MAE(z):", round(full_mae, 4))

# 2) Same tree with min_samples_leaf=5: replicates cannot be split, so we expect the same number of leaves
leaf5_tree = make_tree()
leaf5_tree.set_params(tree__min_samples_leaf=5)
leaf5_tree.fit(X.iloc[train_idx], z.iloc[train_idx])
print("min_sample_leaf = 5, fold 0 | leaves:", leaf5_tree.named_steps["tree"].get_n_leaves())

# 3) Pruning path 
# cost_complexity_pruning_path belongs to the tree alone, so it needs the columns already transformed
# The preprocessor was fitted on these same rows only, inside full_tree
X_train_t = full_tree.named_steps["preprocessing"].transform(X.iloc[train_idx])
path = full_tree.named_steps["tree"].cost_complexity_pruning_path(X_train_t, z.iloc[train_idx])
alphas = path.ccp_alphas    # increasing: alphas[0] = 0 (no pruning), the last one prunes down to the root
print("Pruning path, fold 0 training rows | candidate alphas:", len(alphas),
     "| largest:", round(alphas[-1], 4), "(units of z squared)")

# For each declared value, count how many pruning steps of the path it applies:
# alphas[1:] skips the first value (0, no pruning), and "<= value" marks the steps that happen
for value in param_grid["tree__ccp_alpha"]:
    print("ccp_alpha =", value, "-> pruning steps of the path it applies:", int((alphas[1:] <= value).sum()))

**Fold 0 check.** This cell only describes the data; nothing in it is used to choose the hyperparameters.

- Without any limit, the tree has 1950 leaves, exactly the 1950 configurations in the training rows of fold 0. It puts each training configuration in its own leaf and predicts the mean z of its replicates.
- With `min_samples_leaf` = 5 it has the same 1950 leaves, as expected from the replicates.
- The pruning path of this tree has 1836 candidate alphas; the largest, 0.4316, is the one that would prune it down to the root. The declared values apply 0, 1365, 1671 and 1803 pruning steps. Most steps already happen at very small alphas, which suggests that many neighbouring leaves have almost the same mean z, so merging them costs almost nothing. Even 0.001 is far below 0.4316, so the grid never comes close to a very small tree.
- The validation MAE(z) of the unlimited tree on fold 0 (0.1263) is shown only for reference.

In [ ]:
# GridSearchCV(pipe, param_grid, cv=folds, scoring='neg_mean_absolute_error', return_train_score=True)
# Search object: it does not fit anything yet, it only stores what to try and how to score it
search = GridSearchCV(
    make_tree(),                             # the pipeline to tune; GridSearchCV fits a fresh copy each time
    param_grid,                              # the values declared above
    cv=folds,                                # the shared canonical folds, never rebuilt
    scoring="neg_mean_absolute_error",       # MAE(z) with a minus sign, because GridSearchCV keeps the highest score
    return_train_score=True,                 # also keep training error, to compare it with validation
)

start = time.time()                                        # time before the search
search.fit(X, z)                                           # all fits on the folds, then one final fit on all rows with the winner 
search_seconds = time.time() - start                       # seconds the whole search took
print("Grid search time (s):", round(search_seconds, 1))

In [ ]:
# The selected configuration
best = search.best_params_           # dictionary with the winning value of each hyperparameter
best_index = search.best_index_      # row of the winner in the results table

# Chosen value of each hyperparameter, next to the values that were tried, to see whether the winner lies at the edge of the grid
print("max_depth:", best["tree__max_depth"], "| tried:", param_grid["tree__max_depth"])
print("min_samples_leaf:", best["tree__min_samples_leaf"], "| tried:", param_grid["tree__min_samples_leaf"])
print("ccp_alpha:", best["tree__ccp_alpha"], "| tried:", param_grid["tree__ccp_alpha"])

# best_score_ is the winner's validation score averaged over the 5 folds
val_mae = -search.best_score_                                      # undo the minus sign: positive MAE(z)
# cv_results_ is a table with one position per combination; we read the winner's mean training score
train_mae = -search.cv_results_["mean_train_score"][best_index]    # training MAE(z) of the same configuration
print("Mean validation MAE(z):", round(val_mae, 4))
print("Mean training MAE(z):", round(train_mae, 4))

In [ ]:
# MAE(z) for each of the five folds, its mean and spread, RMSE(z), R2(z)
# and one MAE in nanoseconds computed after 10 ** z_hat

# Refit the selected tree on each fold by hand, to get its predictions and compute every metric
# Empty lists, filled with one value per fold inside the loop 
fold_mae = []        # MAE(z) of each fold
fold_rmse = []       # RMSE(z) of each fold
fold_r2 = []         # R2(z) of each fold
fold_mae_ns = []     # MAE in nanoseconds of each fold
fold_seconds = []    # fitting time of each fold

for f in range(5):
    # Rows of this fold 
    train_idx = folds[f][0]         # training rows of fold f
    val_idx = folds[f][1]           # validation rows of fold f

    # Same model as the winner of the search
    model = make_tree()             # fresh, unfitted pipeline
    model.set_params(               # copy the three winning values
        tree__max_depth=best["tree__max_depth"],
        tree__min_samples_leaf=best["tree__min_samples_leaf"],
        tree__ccp_alpha=best["tree__ccp_alpha"],
    )
    # Fit, timing only the fit
    start = time.perf_counter()      # clock starts before fitting
    model.fit(X.iloc[train_idx], z.iloc[train_idx])    # learn only from the training rows
    fold_seconds.append(time.perf_counter() - start)   # seconds this fit took

    # Predict the validation rows 
    z_hat = model.predict(X.iloc[val_idx])            # predicted z for the validation rows
    z_true = z.iloc[val_idx]                          # real z of the same rows

    # Metrics on the z scale
    mae = mean_absolute_error(z_true, z_hat)            # average absolute error, in log10(ns)
    rmse = np.sqrt(mean_squared_error(z_true, z_hat))   # square root of the average squared error
    r2 = r2_score(z_true, z_hat)                        # share of the variance of z that is explained
    # One metric on the physical scale 
    latency_hat = 10 ** z_hat                           # undo the log10: predictions back to nanoseconds
    mae_ns = mean_absolute_error(latency_ns.iloc[val_idx], latency_hat)  # average absolute error, in ns

    # Store the values of this fold
    fold_mae.append(mae)
    fold_rmse.append(rmse)
    fold_r2.append(r2)
    fold_mae_ns.append(mae_ns)

    # Check: GridSearchCV stored the same MAE(z) for this fold (with a minus sign)
    grid_mae = -search.cv_results_["split" + str(f) + "_test_score"][best_index]

    print("fold", f,
         "| MAE(z):", round(mae, 4),
         "| RMSE(z):", round(rmse, 4),
         "| R2(z):", round(r2, 4),
         "| MAE(ns):", round(mae_ns),
         "| MAE(z) from GridSearchCV:", round(grid_mae, 4))

# Lists to arrays, so we can take means here and subtract the P2 and P3 scores in P4.5
fold_mae = np.array(fold_mae)
fold_rmse = np.array(fold_rmse)
fold_r2 = np.array(fold_r2)
fold_mae_ns = np.array(fold_mae_ns)

# Summary over the five folds 
mae_spread = fold_mae.max() - fold_mae.min()     # spread as R4 defines it: largest minus smallest fold
print("MAE(z) mean:", round(fold_mae.mean(), 4), "| spread:", round(mae_spread, 4),
     "| min:", round(fold_mae.min(), 4), "| max:", round(fold_mae.max(), 4))
print("RMSE(z) mean:", round(fold_rmse.mean(), 4))
print("R2(z) mean:", round(fold_r2.mean(), 4))
print("MAE(ns) mean:", round(fold_mae_ns.mean()))
# An error of MAE(z) in log10 is a factor of 10 ** MAE(z); minus 1 gives it as a relative error
print("Relative error for MAE(z):", round(10 ** fold_mae.mean() - 1, 2))
print("Mean fitting time of one tree (s):", round(np.mean(fold_seconds), 3))
print("Tree MAE(z) per fold (for R4):", np.round(fold_mae, 4))

**Interpretation.** 

*Selected configuration:* The search took less than a minute and chose `max_depth` = 12, `min_samples_leaf` = 5 and `ccp_alpha` = 0.0. Two of these values lie at the edge of the grid, but they are natural limits, not a sign that the grid is too short: no `min_samples_leaf` below 5 gives a different tree, and `ccp_alpha` cannot be lower than 0.0. `max_depth` = 12 lies inside the grid, next to `None`, so the selected tree is limited only by its depth. The mean validation MAE(z) is 0.1306 and the mean training MAE(z) is 0.0324: the tree fits the configurations it was trained on much more closely than new ones.

*Error on the five folds:* The validation MAE(z) goes from 0.1238 to 0.1353, with a mean of 0.1306 and a spread of 0.0115, so it changes little with the configurations held out. Since |z - z_hat| = |log10(predicted/measured)|, a MAE(z) of 0.1306 means that a typical prediction is 10 ** 0.1306 times too high or too low; when it is too high, that is about 35% above the measured latency (relative error 0.35). This is a factor, not an error in nanoseconds. RMSE(z) = 0.2046 is clearly larger than MAE(z), so some configurations have much larger errors than the typical one. R2(z) = 0.9405: the tree explains most of the variation of z.

*Error in nanoseconds:* After 10 ** z_hat the mean MAE is 14073675 ns. Absolute errors grow with the latency itself, so this number is dominated by the slowest configurations, as in Practice 2. The two metrics can therefore order the folds differently: fold 0 has the largest MAE in ns (18018976) and the smallest MAE(z) (0.1238). MAE(z) stays the primary metric because the same relative error costs the same at every latency level.

In [ ]:
# Only to print the cyber sizes used in the Markdown (in the final notebook R1's cyber cells print them)
cyber_dev = pd.read_csv("../datos/cyber/cyber_development.csv")
cyber_groups = pd.read_csv("../datos/cyber/cyber_groups.csv")
print("Cyber rows:", len(cyber_dev), "| cyber feature groups:", cyber_groups["feature_group"].nunique())

**What would have gone wrong had the classification hyperparameters been reused.** 

- *Sample size:* cyber has 34928 rows and 34883 feature groups, so almost every row is a different pattern. Storage has 12190 rows but only 2438 different configurations. In cyber a leaf of 5 rows holds about 5 different patterns; in storage it holds only 1 configuration. The same `min_samples_leaf` therefore gives storage leaves with far fewer independent observations.
- *Structure of the rows:* storage rows come in groups of 5 identical replicates. Any `min_samples_leaf` below 5 does nothing here, and leaf sizes must be read in configurations, not rows.
- *Impurity minimised:* the classification tree minimises the Gini impurity, which measures how mixed the classes are in a node. The regression tree minimises the squared error of z, whose scale is set by the variance of z, 0.7056. A depth that is enough to separate classes need not be enough to follow a continuous latency.
- *Units of ccp_alpha:* in classification ccp_alpha is measured in units of class impurity, here it is in squared units of z. The same number means a different amount of pruning in each task.

# P4.4 — Rules

In [ ]:
# Say which fitted tree the rules come from (the selected one, or a shallower tree for reading only)

# The rules come from the selected tree itself: GridSearchCV refitted it on all the development rows
final_pipe = search.best_estimator_                 # preprocessing + tree, already fitted
final_tree = final_pipe.named_steps["tree"]         # the DecisionTreeRegressor inside the pipeline
# Names of the columns the tree sees
feature_names = final_pipe.named_steps["preprocessing"].get_feature_names_out()
X_all_t = final_pipe.named_steps["preprocessing"].transform(X)       # all the rows, as the tree sees them
leaf_of_row = final_tree.apply(X_all_t)             # number of the leaf where each row ends

print("Rules come from: the selected tree (best_estimator_), not a shallower tree")
print("Rows used to fit it:", final_tree.tree_.n_node_samples[0])    # node 0 is the root: it holds every row
print("Depth:", final_tree.get_depth(), "| leaves:", final_tree.get_n_leaves())

In [ ]:
# The full rule set of two named leaves, each with its node, predicted value
# and number of training rows (required figure)

tree = final_tree.tree_         # arrays of the fitted tree, one position per node
# Arrays used below (one value per node):
#   children_left / children_right: number of the left/right child (-1 if the node is a leaf)
#   feature / threshold: column tested in the node and its threshold (column <= threshold goes left)
#   n_node_samples: training rows that reach the node
#   value[node, 0, 0]: prediction of the node = mean z of its training rows
MIN_ROWS = 20                   # only leaves with at least 20 rows = 4 configurations

# Choose the two leaves
# Node numbers that are leaves AND have at least MIN_ROWS training rows
big_leaves = np.flatnonzero((tree.children_left == -1) & (tree.n_node_samples >= MIN_ROWS))
leaf_z = tree.value[big_leaves, 0, 0]     # predicted z of each of those leaves
fast_leaf = big_leaves[np.argmin(leaf_z)] # argmin = position of the lowest z -> fastest leaf
slow_leaf = big_leaves[np.argmax(leaf_z)] # argmax = position of the highest z -> slowest leaf
print("Leaves with at least", MIN_ROWS, "rows:", len(big_leaves))
print()

# print the path from the root to one leaf
def print_rule(leaf, name):
    # Take one training row that ends in this leaf: following its path from the root gives the rule
    x = X_all_t[leaf_of_row == leaf][0]     # any training row that ends in this leaf
    print(name, "| node", leaf)
    node = 0      # start at the root
    while tree.children_left[node] != -1:   # repeat until we reach a leaf
        # Column tested here, without the "cat__" / "num__" prefix of the preprocessor
        column = feature_names[tree.feature[node]].split("__")[-1]
        # The same test the tree applies: True means the row goes to the left child 
        goes_left = x[tree.feature[node]] <= tree.threshold[node]
        if column in NUMERIC:
            # Numeric column: print the threshold and the side the row takes
            threshold = round(tree.threshold[node],3)
            if goes_left:
                print("    ", column, "<=", threshold)
            else:
                print("    ", column, ">", threshold)
        else:
            # one-hot column: "<= 0.5" means the row does NOT have this level
            for nominal in NOMINAL:
                if column.startswith(nominal + "_"):
                    level = column[len(nominal) + 1:] # what comes after "raid_"
                    if goes_left:
                        print("    ", nominal, "is not", level)
                    else:
                        print("    ", nominal, "is", level)
        # Move to the child the row goes to
        if goes_left:
            node = tree.children_left[node]
        else:
            node = tree.children_right[node]
    assert node == leaf # the manual walk must end where apply() put the row
    z_leaf = tree.value[leaf][0][0]  # prediction on the leaf, in log10(ns)
    print("   predicted z:", round(z_leaf, 4), "| latency = 10 ** z:", round(10 ** z_leaf), "ns")
    print("   training rows:", tree.n_node_samples[leaf], "| configurations:", tree.n_node_samples[leaf] // 5)
    # What the training rows of this leaf really contain, in the original columns
    rows_in_leaf = X[leaf_of_row == leaf]  # same row order as X_all_t
    for col in NOMINAL:
        print("    ", col, "in this leaf:", sorted(rows_in_leaf[col].unique()))
    for col in NUMERIC:
        print("    ", col, "in this leaf: from", rows_in_leaf[col].min(), "to", rows_in_leaf[col].max())
    print()

print_rule(fast_leaf, "Leaf A (fastest leaf with at least " + str(MIN_ROWS) + " rows)")
print_rule(slow_leaf, "Leaf B (slowest leaf with at least " + str(MIN_ROWS) + " rows)")

**Which tree and which leaves.** The rules come from the selected tree itself (`best_estimator_`), which `GridSearchCV` refitted on all 12190 development rows; no shallower tree was fitted just for reading. It has depth 12 and 1456 leaves, far too many to read as a whole, so we show the complete rule of two leaves. To avoid leaves that describe a single configuration, we only consider leaves with at least 20 training rows (4 configurations). This reading rule was fixed before looking at the leaves and does not depend on any score; 119 leaves pass it. Leaf A has the lowest predicted z and Leaf B the highest, so they show the two extremes of the tree, not typical leaves.

**How to read the figure.** Each line is one test on the path from the root to the leaf, in order. Numeric thresholds lie halfway between two observed values. For a one-hot column, the tree tests whether the level is present, printed as "is" or "is not". Some variables are tested more than once with a tighter bound each time (`block_size <= 48.0`, `<= 24.0`, `<=12.0` in Leaf A); only the tightest one matters. The predicted z of a leaf is the mean z of its training rows, so 10 ** z is the geometric mean of their latencies.

- **Leaf A** (node 78, 30 training rows, 6 configurations): random access (`load_type` is not sequential), write, a RAID scheme other than 4+1, `n_jobs <= 5.0`, `iodepth` above 9.0 and at most 16.5, `block_size <= 12.0`, `read_fraction > 21.5` and `n_disks <= 19.0`. Predicted latency: 228877 ns.

**One rule read as an operations statement (Leaf B: node 2214, 20 training rows, 4 configurations).** On an hdd array in RAID 4+1 with more than 4.5 and at most 13.0 disks, a sequential write workload with more than 7.5 jobs, a queue depth above 12.5 and at most 29.0, and a block size up to 384.0 is predicted to have a latency of about 436246220 ns, just under half a second. That is more than three orders of magnitude above Leaf A (z = 8.6397 against z = 5.3596).

This statement summarises only 4 training configurations. It describes an association in the fitted tree, not a cause, and it only holds inside the ranges seen in training. A leaf predicts the same value for every configuration it contains, so the rule cannot say how latency changes within these ranges.

# P4.5 — Challenge B against Practice 3 and Practice 2

In [ ]:
# five per-fold primary scores of one earlier row of comparativa.csv
def previous_scores(milestone, challenge, family):
    table = pd.read_csv("../comparativa.csv")
    # keep only the row with this milestone, challenge and family
    row = table[(table["milestone"] == milestone) & (table["challenge"] == challenge) & (table["family"] == family)]
    # the five scores are stored as one text, e.g. "0.9436 0.9473 0.9490 0.9428 0.9451"
    text = row["primary_metric_per_fold"].iloc[0]
    # cut the text at the spaces and turn each piece into a number
    return np.array(text.split(), dtype=float)

In [ ]:
p2_storage = previous_scores("P2", "storage", "KNN")
p3_storage = previous_scores("P3", "storage", "LinearRegression")
print("P2 KNN:   ", p2_storage)
print("P3 linear:", p3_storage)

# check of the method: P2 - P3 must give the differences printed in Practice 3 (mean -0.0181)
# for MAE(z) lower is better, so delta = previous - new
deltas = p2_storage - p3_storage
print("P2 - P3:  ", np.round(deltas, 4))
print("mean:", round(np.mean(deltas), 4), "| spread:", round(spread(deltas), 4))

# R3's five MAE(z) of the selected tree (from R3's notebook)
# delta = previous - tree: for MAE(z) lower is better, so a positive delta means the tree did better
tree_storage = np.array([0.1238, 0.1286, 0.1319, 0.1353, 0.1332])

p3_diff = p3_storage - tree_storage
p2_diff = p2_storage - tree_storage

print("tree:     ", tree_storage, "| mean:", round(np.mean(tree_storage), 4), "| spread:", round(spread(tree_storage), 4))
print()
print("P3 - tree:", np.round(p3_diff, 4))
print("mean:", round(np.mean(p3_diff), 4), "| spread:", round(spread(p3_diff), 4), "| folds where the tree is better:", np.sum(p3_diff > 0), "of 5")
print("spread of P3 linear:", round(spread(p3_storage), 4))
print()
print("P2 - tree:", np.round(p2_diff, 4))
print("mean:", round(np.mean(p2_diff), 4), "| spread:", round(spread(p2_diff), 4), "| folds where the tree is better:", np.sum(p2_diff > 0), "of 5")
print("spread of P2 KNN:", round(spread(p2_storage), 4))

**Reading of the differences, and which kind of comparison this is.**

*Against the Practice 3 linear regression.* The tree is better on all five folds. The mean paired difference is +0.0676 in MAE(z), with a range of 0.0209 across the five differences. Both models have a much smaller fold-to-fold range (0.0115 for the tree and 0.011 for the linear model), so the change is large relative to the variability we observed.

*Against the Practice 2 KNN.* The tree is also better on all five folds. The mean paired difference is +0.0495, with a range of 0.0162, again large next to the fold-to-fold range of each model (0.0115 for the tree and 0.0108 for the KNN).

Both readings describe these five folds only. They are not a significance test, and they do not establish that a difference would hold on new data.

*Which kind of comparison.* Both are **pipeline comparisons**. The tree keeps every one-hot level and uses the numeric predictors unscaled. The Practice 3 model dropped one reference level per nominal predictor and scaled the numeric predictors; the Practice 2 KNN kept every level but scaled them. Scaling would not change a tree's partitions, but we did not rerun the earlier families under the tree's representation, so the evidence supports only that this pipeline did better than the earlier pipelines on these folds.

In [ ]:
# Numbers of the earlier storage models, read from comparativa.csv, used in the answers below
table = pd.read_csv("../comparativa.csv")
storage_rows = table[table["challenge"] == "storage"]
for i in range(len(storage_rows)):
    row = storage_rows.iloc[i]
    print(row["milestone"], row["family"],
         "| MAE(z):", row["primary_metric"],
         "| spread:", row["primary_metric_spread"],
         "| fit time (s):", row["fit_time_s"])
    print("     ", row["supporting_metrics"])

# Share of the Dummy MAE(z) that each model removes (1 = perfect, 0 = no better than the Dummy)
dummy_mae = previous_scores("P2", "storage", "Dummy").mean()
print("Share of the Dummy MAE(z) removed | tree:", round(1 - fold_mae.mean() / dummy_mae, 3),
     "| P2 KNN:", round(1 - p2_storage.mean() / dummy_mae, 3),
     "| P3 linear:", round(1 - p3_storage.mean() / dummy_mae, 3))

# Cyber: share of the gap between Dummy and a perfect macro F1(= 1) that each model closes
cyber_rows = table[table["challenge"] == "cyber"]   # keep only the cyber rows
dummy_f1 = previous_scores("P2", "cyber", "Dummy").mean() # mean macro F1 of the cyber Dummy
for i in range(len(cyber_rows)):
    row = cyber_rows.iloc[i]
    f1 = previous_scores(row["milestone"], "cyber", row["family"]).mean() # mean macro F1 over the 5 folds
    share = (f1 - dummy_f1) / (1 - dummy_f1)
    print("Cyber,", row["milestone"], row["family"], "| macro F1:", round(f1,4),
         "| share of the gap closed:", round(share, 3))

### The six fixed comparison questions

Answer each one for both comparisons: the tree against the Practice 3 linear regression and against the Practice 2 KNN.

1. **Size and consistency of the change.** Against the Practice 3 linear regression, the tree has a lower MAE(z) on all 5 folds: differences 0.0809, 0.0651, 0.0665, 0.06 and 0.0655, mean 0.0676, spread 0.0209. Against the Practice 2 KNN it is also better on all 5 folds: differences 0.0578, 0.0584, 0.0459, 0.0422 and 0.043, mean 0.0495, spread 0.0162; the gain is smaller on the last three folds. This only describes these five folds; it is not a statistical test. The tree and the KNN were both tuned on these same folds, so both scores are slightly optimistic, the tree probably a bit more because its search was larger.
2. **Route and representation.** All three models use the primary route, the same 9 source predictors, the same target z and the same canonical folds, but not the same representation: Practice 2 uses full one-hot with `StandardScaler`, Practice 3 drops one reference level per nominal predictor and also scales, and Practice 4 uses full one-hot with unscaled numeric columns. Strictly, both comparisons are pipeline comparisons. The level dropped in Practice 3 is, here, a column of its own; in Practice 3 it was the case where all the other columns of that predictor are 0.

   - Against Practice 2, the only difference is the scaler. Rescaling each column with a positive factor keeps the order of its values, so it keeps the partition that every threshold induces: fitted on the scaled columns, the tree would be the same tree, apart from the threshold numbers (and tie-breaking and finite precision). We did not refit it to check. This comparison is therefore close to a family comparison: on the same encoded columns, the tree did better than KNN on these folds.
   - Against Practice 3, the encoding also differs. With the reference level dropped, a test such as "raid is 4+1" (used in Leaf B) is not a single column and needs several splits. The evidence supports "this tree pipeline did better than that linear pipeline", not that the family alone explains the whole difference.

3. **What can the new family express?** Against the linear model: interactions and thresholds. The linear model on z adds a fixed amount per predictor, whatever the other predictors are. The tree's prediction depends on combinations (Leaf B only applies when sequential access, hdd, write and RAID 4+1 hold together), and it can place a threshold where latency jumps. Against KNN: KNN can also follow interactions locally, but its distance gives the same weight to every standardised predictor and it always averages the same number of neighbours. The tree chooses which predictors matter in each region, ignores the others there, and gives explicit rules. What the tree cannot express is a smooth change: its prediction is piecewise constant, with at most 1456 different values, and it stays constant outside the ranges seen in training.
4. **What did it cost?** Fitting time is small for all three: 0.06 s per fold for the tree, against 0.0298 s for the linear model and 0.0127 s for KNN. These were measured in different runs, so only the order of magnitude matters. Tuning effort is clearly higher: 96 combinations and 480 fits, about half a minute in total, against no search for the linear model (Ridge and Lasso were explored separately in Practice 3) and a one-dimensional grid of K for KNN. Interpretability improves against KNN, which gives no rules at all. Against the linear model the change is mixed: each path can be read, but it has 12 conditions, and the whole tree, with 1456 leaves, cannot be read at once, while the linear model is summarised by one coefficient per column.
5. **Did the ordering of the two challenges' difficulty change?** We compare the two challenges by the share of the gap between the Dummy and a perfect score that each model closes. In storage this is 1 - MAE(z)/MAE_Dummy(z), because a perfect MAE(z) is 0. In cyber it is (F1 - F1_Dummy)/(1 - F1_Dummy), where the Dummy's macro F1 is 0.0242. Both measures are 0 for the Dummy and 1 for a perfect model, so they can be placed side by side even though they come from different metrics.
- Practice 2 (KNN): cyber 0.944 (macro F1 0.9456), storage 0.734
- Practice 3 (linear models): cyber 0.834 (logistic regression, macro F1 0.8381), storage 0.707 (linear regression).
- Practice 4 (tree): cyber 0.944, storage 0.807.
Conclusion: the ordering did not change. The tree is the best model so far in storage and narrows the gap (0.807 against 0.734 for the KNN), but it still falls short of the best cyber model (KNN, 0.944) and even of the cyber logistic regression (0.834). Any tree result in cyber can only keep or raise cyber's best share, so cyber remains the challenge where the models come closest to a perfect score.

6. **Would you ship the new model?** Yes, as the Challenge B model, replacing the KNN. It has the lowest MAE(z) on all five folds, and, unlike in Practice 3, no supporting metric disagrees: RMSE(z) 0.2046 against 0.2568 (KNN) and 0.2703 (linear), R2(z) 0.9405 against 0.9065 and 0.8964, and MAE in ns 14073675 against 17175412 and 16289769. Caveats: the gap between training and validation MAE(z) is large (0.0324 against 0.1306), the configuration was chosen on the same folds that measure it, a single tree can change a lot with small changes in the data, and it cannot extrapolate beyond the training ranges. We would change our mind if the result on the final test were clearly worse, if a family from a later practice did better on these folds, or if we needed predictions for configurations outside the ranges seen in training.

## Comparison table

In [ ]:
# the Practice 4 storage row, appended at the end of comparativa.csv
# the P2 and P3 rows are never rewritten: mode="a" only adds lines at the end of the file
table = pd.read_csv("../comparativa.csv")

storage_row = {
    "milestone": "P4", "challenge": "storage", "family": "DecisionTree", "route": "primary",
    "configuration": "criterion=squared_error, max_depth=12, min_samples_leaf=5, ccp_alpha=0.0; selected by lowest mean MAE(z) "
                     "from max_depth [4, 6, 8, 10, 12, None], min_samples_leaf [5, 10, 20, 40], ccp_alpha [0, 1e-05, 0.0001, 0.001]",
    "preprocessing": "one-hot on the 4 nominal predictors with every level kept; numeric predictors unchanged (no scaler); fitted inside each fold; target z = log10(latency_ns)",
    "features": "features.md: the 9 distributed source predictors (frozen basis)",
    "validation": "canonical GroupKFold(5) on config_group, read from folds/storage_folds.csv; grid search with cv=folds",
    "primary_metric_per_fold": " ".join(f"{v:.4f}" for v in tree_storage),
    "primary_metric": round(np.mean(tree_storage), 4),
    "primary_metric_spread": round(spread(tree_storage), 4),
    "supporting_metrics": "RMSE(z) 0.2046; R2(z) 0.9405; MAE_ns 14073675 (computed after inverting predictions)",
    "fit_time_s": round(np.mean(fold_seconds), 4),
    "interpretability": "rules from root to leaf, read on the physical scale with 10^z",
    "notes": "non-linear model with its own grid; better than the P2 KNN and the P3 linear model on every fold",
}

for row in [storage_row]:
    # the row must have exactly the columns of the table, in the same order
    assert list(row.keys()) == list(table.columns)
    # if the notebook is run again, do not add the same row twice
    already = (table["milestone"] == row["milestone"]) & (table["challenge"] == row["challenge"]) & (table["family"] == row["family"])
    if already.any():
        print("already in comparativa.csv:", row["milestone"], row["challenge"], row["family"])
    else:
        pd.DataFrame([row]).to_csv("../comparativa.csv", mode="a", header=False, index=False)
        print("added to comparativa.csv:", row["milestone"], row["challenge"], row["family"])

# the whole table after the append: P2 and P3 rows unchanged, P4 rows at the end
final_table = pd.read_csv("../comparativa.csv")
print(final_table[["milestone", "challenge", "family", "route", "primary_metric", "primary_metric_spread"]].to_string(index=False))